In [1]:
print(123)

123


In [3]:
import sys
sys.path.append('..')

In [ ]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]
#this gives us an array of JSON containing the content of the page and the filename for each

In [4]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [ ]:
###
# For each page, build a JSON user prompt from its filename and content, 
# then call llm_structured with the Questions model. 
# Turn each returned question into 
# a record labeled with the page's filename. The call also returns the token usage, 
# the same as in the lessons.
###


In [8]:
doc1 = documents[0]
doc1

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

In [6]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [7]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [ ]:
import json

#just check the first page to start
user_prompt = json.dumps(doc1)

In [13]:
user_prompt

'{"content": "# Introduction\\n\\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\\n\\nIn this module, we\'ll build a working Retrieval-Augmented\\nGeneration (RAG) system from scratch, step by step.\\n\\nWe write everything in plain Python. We build a small search index by\\nhand and call the LLM ourselves. I want you to see every piece first.\\nThat way you know what a framework does for you before you reach for\\none.\\n\\nPlaces where you can find me:\\n\\n- [My substack](https://alexeyondata.substack.com/)\\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\\n- [X](https://x.com/Al_Grigor)\\n\\n## LLMs\\n\\nAn LLM (Large Language Model) is a neural network trained on massive\\namounts of text. Given a prompt, it generates a continuation - a\\nplausible next piece of text.\\n\\nThink of your phone. When you type \\"how are\\" in WhatsApp, it suggests\\n\\"you\\" as the next word. \\"How are you\\" is the most common

In [14]:
from evaluation_utils import llm_structured

In [ ]:
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)


In [16]:
print(result.questions)
print(usage)

['What is RAG supposed to do for an LLM, and why do people use it so often in real projects?', 'What are the main limits of an LLM that this module says RAG can help with?', 'How will the course build the RAG system in this module—what pieces are you making from scratch in Python?', 'What kind of example app are they building to show RAG in practice?', 'What’s the difference between the first part of the module and the second part?']
ResponseUsage(input_tokens=1020, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=113, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1133)


In [17]:
first3 = documents[:3]

In [20]:
from evaluation_utils import llm_structured_retry

In [108]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["filename"]
        })

    return results, usage

In [24]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(first3):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/3 [00:00<?, ?it/s]

In [26]:
import pandas as pd

df_usages = pd.DataFrame(usages)

In [ ]:
#answer Q1 is 1400
df_usages

,0,1,2,3,4
0,"(input_tokens, 1020)","(input_tokens_details, InputTokensDetails(cach...","(output_tokens, 105)","(output_tokens_details, OutputTokensDetails(re...","(total_tokens, 1125)"
1,"(input_tokens, 1286)","(input_tokens_details, InputTokensDetails(cach...","(output_tokens, 115)","(output_tokens_details, OutputTokensDetails(re...","(total_tokens, 1401)"
2,"(input_tokens, 1753)","(input_tokens_details, InputTokensDetails(cach...","(output_tokens, 104)","(output_tokens_details, OutputTokensDetails(re...","(total_tokens, 1857)"


In [29]:
import pandas as pd

df_ground_truth = pd.read_csv("ground-truth.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [109]:
ground_truth[:10]

[{'question': "What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?",
  'filename': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'Why does this course build the RAG project in plain Python instead of starting with a framework or library?',
  'filename': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'What are the main weaknesses of large language models that this module is trying to work around?',
  'filename': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'What will the course build in the first part of the module, and how is the second part different?',
  'filename': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'What kind of example app are you building here, and what data will it answer questions from?',
  'filename': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'What do I need installed before starting this module?',
  'filename': '01-agentic-rag/lessons/02-environmen

In [40]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [110]:
chunks[:10]

[{'start': 0,
  'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour ph

In [47]:
#grab the content only from the chunks and put them in a list
texts = [chunk["content"] for chunk in chunks]

In [52]:
from embedder import Embedder
from minsearch import VectorSearch
from minsearch import Index

embed = Embedder()




In [59]:
#use minsearch to index the documents. Use the filename as a keyword field and the content as a text field.
index = Index(
        text_fields=["content"],
        keyword_fields=["filename"]
    )
index.fit(chunks)

In [61]:
def text_search(query, num_results=5):

    return index.search(
        query,
        num_results = num_results
    )

In [53]:
#go through each of the content in batches and encode them into vectors. Put them into the X list
#then use numpy to turn X into an array
from tqdm.auto import tqdm
import numpy as np

batch_size = 50
X = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = embed.encode_batch(batch)
    X.extend(batch_vectors)

X = np.array(X)

  0%|          | 0/6 [00:00<?, ?it/s]

In [68]:
X.shape

(295, 384)

In [54]:
#minsearch will compute the dot product for you, and will also allow you to filter by keyword fields

vindex = VectorSearch(keyword_fields=["filename"])
vindex.fit(X, chunks)

In [77]:
def vector_search(query, num_results=5):
    query_vector = embed.encode(query)
    return vindex.search(query_vector, num_results=num_results)

In [57]:
gtq1 = ground_truth[0]["question"]
gtq1

"What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?"

In [62]:
text_results = text_search(gtq1)

In [ ]:
#Q2 answer = 01-agentic-rag/lessons/03-rag.md
text_results

[{'start': 3000,
  'content': 'we drop it.\n\nBuild a prompt that includes both the question and the context:\n\n```python\nprompt = f"""\nYour task is to answer questions from the course participants\nbased on the provided context.\n\nUse the context to find relevant information and provide accurate\nanswers. If the answer is not found in the context,\nrespond with "I don\'t know."\n\nQuestion:\n{question}\n\nContext:\n{context}\n"""\n```\n\nInstead of sending the raw question to the LLM, we send this prompt:\n\n```python\nanswer = llm(prompt)\nprint(answer)\n```\n\nAfter that, the answer is correct: "Yes, you can still join. If you want to\nreceive a certificate, you need to submit your project while\nsubmissions are still open."\n\nThis is the answer we actually want to give to our students. What we\njust did is nothing but RAG.\n\n## Retrieval plus generation\n\nRAG stands for Retrieval-Augmented Generation. Generation is the LLM\nproducing text, and retrieval is search. We retriev

In [78]:
vector_results = vector_search(gtq1)

In [ ]:
#Q3 answer = 01-agentic-rag/lessons/01-intro.md
vector_results

[{'start': 0,
  'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour ph

In [81]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [82]:
def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

In [83]:
hybrid_results = hybrid_search(gtq1)

In [84]:
hybrid_results

[{'start': 0,
  'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour ph

In [95]:
def compute_relevance_text(q):
    doc_id = q["filename"]
    results = text_search(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["filename"] == doc_id))

    return relevance

In [96]:
def compute_relevance(q, search_function):
    doc_id = q["filename"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["filename"] == doc_id))

    return relevance

In [97]:
def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [98]:
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

In [99]:
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break

    return total_score / len(relevance)

In [ ]:
def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

In [107]:
#question 4 answer hit rate = 0.758333
evaluate(
    ground_truth,
    text_search
)

  0%|          | 0/360 [00:00<?, ?it/s]

[[0, 0, 0, 0, 1], [1, 0, 1, 0, 0], [1, 1, 0, 0, 1], [1, 0, 1, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 1, 0, 0, 0], [0, 0, 0, 0, 0], [1, 0, 0, 0, 0], [1, 1, 1, 0, 0], [1, 1, 0, 0, 0], [0, 1, 0, 0, 1], [0, 0, 0, 0, 0], [0, 1, 1, 0, 0], [0, 0, 0, 0, 0], [1, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [1, 1, 0, 0, 0], [1, 0, 0, 0, 0], [1, 1, 1, 0, 0], [0, 0, 1, 1, 0], [0, 1, 1, 0, 0], [1, 1, 0, 0, 1], [1, 1, 0, 0, 0], [0, 0, 0, 0, 0], [0, 1, 0, 1, 0], [0, 0, 0, 0, 0], [1, 1, 0, 0, 0], [1, 0, 1, 1, 0], [1, 0, 0, 0, 0], [1, 1, 0, 0, 1], [0, 0, 0, 0, 0], [0, 1, 1, 1, 0], [1, 0, 1, 1, 0], [0, 0, 0, 0, 0], [1, 0, 0, 0, 1], [1, 1, 0, 0, 0], [0, 0, 0, 0, 0], [1, 0, 0, 0, 0], [1, 1, 0, 0, 0], [1, 1, 1, 1, 0], [1, 1, 1, 1, 0], [1, 1, 1, 1, 1], [1, 1, 1, 1, 1], [0, 0, 0, 1, 0], [0, 0, 0, 0, 0], [0, 1, 0, 0, 0], [1, 0, 0, 1, 0], [1, 1, 0, 0, 0], [1, 0, 0, 0, 0], [1, 0, 0, 0, 0], [1, 0, 0, 0, 0], [1, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 1, 0, 0, 0], [1, 0, 1, 0, 0], [0, 0, 1, 0, 0], [0, 0, 0, 0, 

{'hit_rate': 0.7583333333333333, 'mrr': 0.5942592592592594}

In [ ]:
#question 5 answer MRR = 0.5486111
evaluate(
    ground_truth,
    vector_search
)

  0%|          | 0/360 [00:00<?, ?it/s]

{'hit_rate': 0.725, 'mrr': 0.5486111111111112}

In [ ]:
#answer for Q6 is k=1 with MRR=0.648 while the others are 0.6379
for k in [1, 50, 100, 200]:
    results = evaluate(
        ground_truth,
        lambda query, k=k: hybrid_search(query, k=k)
    )
    print(f"k={k}: {results}")

  0%|          | 0/360 [00:00<?, ?it/s]

k=1: {'hit_rate': 0.8388888888888889, 'mrr': 0.6481944444444449}


  0%|          | 0/360 [00:00<?, ?it/s]

k=50: {'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667}


  0%|          | 0/360 [00:00<?, ?it/s]

k=100: {'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667}


  0%|          | 0/360 [00:00<?, ?it/s]

k=200: {'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667}
